In [ ]:
import torch
from pathlib import Path
from omegaconf import OmegaConf
from hydra import compose, initialize_config_dir
import time

from il.dataset import create_dataloaders
from accelerate import Accelerator
from il.model import Scaled
from il.model import Conditional2DUNet
from il.utils.diffusion import ScheduleDDPM, training_loop, ScheduleLogLinear, samples
from torch_ema import ExponentialMovingAverage as EMA

conf_dir = (Path.cwd() / "../src/il/conf").resolve()
if conf_dir is None:
    raise FileNotFoundError("未找到 Hydra 配置目录 src/il/conf")

with initialize_config_dir(version_base=None, config_dir=str(conf_dir)):
    df_cfg = compose(config_name="config")

data_dir = (Path.cwd() / "../data").resolve()
df_cfg.data.data_dirs = [f"{data_dir}/20260428_114003/preprocessed"]

log_dir = (Path.cwd() / "../log").resolve()
# df_cfg.log.save_dir = f"{log_dir}/il_tutorial_{time.strftime('%Y%m%d_%H%M%S')}"
df_cfg.log.save_dir = f"{log_dir}/df_tutorial"

print(f"数据目录: {df_cfg.data.data_dirs}")
print(f"道路采样点数: {df_cfg.data.road_points}")
print(f"车道分隔线数: {df_cfg.data.num_lane_dividers}")
print(f"batch_size: {df_cfg.data.batch_size}")
print(f"train/val/test 比例: {df_cfg.data.train_ratio}/{df_cfg.data.val_ratio}/{1 - df_cfg.data.train_ratio - df_cfg.data.val_ratio:.1f}")

In [ ]:
in_dim = 4
in_ch = 1
out_ch = 1
epochs = 1000
sample_batch_size=64

a = Accelerator()
train_loader, val_loader, test_loader = create_dataloaders(df_cfg)
schedule = ScheduleDDPM(beta_start=0.0001, beta_end=0.002, N=1000)
model = Scaled(Conditional2DUNet)(in_dim=in_dim, in_ch=in_ch, out_ch=out_ch).to(df_cfg.device)

ema = EMA(model.parameters(), decay=0.9999)
ema.to(a.device)
for ns in training_loop(train_loader, model, schedule, epochs=epochs, lr=2e-4, accelerator=a):
    ns.pbar.set_description(f'Loss={ns.loss.item():.5}')